# Build Plant Disease Resource Vector DB

This notebook loads the downloaded Markdown resources, chunks them with overlap, embeds each chunk with a local SentenceTransformer model, and persists a Chroma vector database.

In [1]:
%pip install -U chromadb sentence-transformers langchain-chroma langchain-community langchain-huggingface langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


PROJECT_DIR = Path.cwd().parent
RESOURCES_DIR = PROJECT_DIR / "datasets" / "resources"
INDEX_PATH = RESOURCES_DIR / "index.json"
PERSIST_DIR = Path.cwd() / "chroma_db"

COLLECTION_NAME = "plant_disease_resources"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
MIN_CHUNK_CHARS = 100

print("Project directory:", PROJECT_DIR)
print("Resource index:", INDEX_PATH)
print("Chroma persist directory:", PERSIST_DIR)

Project directory: /work/AAI_project
Resource index: /work/AAI_project/datasets/resources/index.json
Chroma persist directory: /work/AAI_project/rag/chroma_db


In [3]:
with open(INDEX_PATH, encoding="utf-8") as file:
    resource_index = json.load(file)

successful_resources = [record for record in resource_index if record.get("status") == "success"]

print(f"Total records: {len(resource_index)}")
print(f"Successful resources: {len(successful_resources)}")

pd.DataFrame(successful_resources)[
    ["id", "labels", "content_type", "document_file", "extracted_characters"]
].head()

Total records: 60
Successful resources: 60


,id,labels,content_type,document_file,extracted_characters
0,resource_001,[General plant pathology book 1],application/pdf,documents/resource_001_general-plant-pathology...,35581
1,resource_002,[General plant pathology book 2],application/pdf,documents/resource_002_general-plant-pathology...,30964
2,resource_003,[Practical disease management handbook],text/html,documents/resource_003_practical-disease-manag...,31608
3,resource_004,[Apple],text/html,documents/resource_004_apple.md,17882
4,resource_005,[Blueberry],text/html,documents/resource_005_blueberry.md,15590


In [4]:
def flatten_metadata(record: dict) -> dict:
    labels = record.get("labels", [])
    return {
        "resource_id": record["id"],
        "source_url": record.get("url", ""),
        "final_url": record.get("final_url", ""),
        "labels": ", ".join(labels),
        "labels_json": json.dumps(labels),
        "title": record.get("title", ""),
        "document_file": record.get("document_file", ""),
        "content_type": record.get("content_type", ""),
        "retrieval_method": record.get("retrieval_method", ""),
    }


documents = []
for record in successful_resources:
    document_path = RESOURCES_DIR / record["document_file"]
    text = document_path.read_text(encoding="utf-8")
    documents.append(Document(page_content=text, metadata=flatten_metadata(record)))

print(f"Loaded documents: {len(documents)}")
print(f"Total characters: {sum(len(document.page_content) for document in documents):,}")
documents[0].metadata

Loaded documents: 60
Total characters: 984,286


{'resource_id': 'resource_001',
 'source_url': 'https://uwyoextension.org/psep/wp-content/uploads/2012/09/MP-27.pdf',
 'final_url': 'https://uwyoextension.org/psep/wp-content/uploads/2012/09/MP-27.pdf',
 'labels': 'General plant pathology book 1',
 'labels_json': '["General plant pathology book 1"]',
 'title': 'resource_001_general-plant-pathology-book-1',
 'document_file': 'documents/resource_001_general-plant-pathology-book-1.md',
 'content_type': 'application/pdf',
 'retrieval_method': 'direct'}

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " ", ""],
)

chunks = []
skipped_short_chunks = 0
for document in documents:
    split_documents = text_splitter.split_documents([document])
    kept_chunk_index = 0
    for chunk in split_documents:
        if len(chunk.page_content.strip()) < MIN_CHUNK_CHARS:
            skipped_short_chunks += 1
            continue
        chunk.metadata = dict(chunk.metadata)
        chunk.metadata["chunk_index"] = kept_chunk_index
        chunk.metadata["chunk_id"] = f"{chunk.metadata['resource_id']}_chunk_{kept_chunk_index:04d}"
        chunks.append(chunk)
        kept_chunk_index += 1

chunk_lengths = [len(chunk.page_content) for chunk in chunks]
print(f"Chunks: {len(chunks)}")
print(f"Skipped short chunks (<{MIN_CHUNK_CHARS} chars): {skipped_short_chunks}")
print(f"Min chunk chars: {min(chunk_lengths)}")
print(f"Mean chunk chars: {sum(chunk_lengths) / len(chunk_lengths):.0f}")
print(f"Max chunk chars: {max(chunk_lengths)}")

pd.DataFrame(
    {
        "resource_id": [chunk.metadata["resource_id"] for chunk in chunks],
        "labels": [chunk.metadata["labels"] for chunk in chunks],
        "chunk_length": chunk_lengths,
    }
).groupby(["resource_id", "labels"]).agg(chunks=("chunk_length", "count"), avg_chars=("chunk_length", "mean")).reset_index().head(10)

Chunks: 1396
Skipped short chunks (<100 chars): 78
Min chunk chars: 102
Mean chunk chars: 774
Max chunk chars: 999


,resource_id,labels,chunks,avg_chars
0,resource_001,General plant pathology book 1,47,897.276596
1,resource_002,General plant pathology book 2,41,837.463415
2,resource_003,Practical disease management handbook,45,813.644444
3,resource_004,Apple,26,785.038462
4,resource_005,Blueberry,22,800.181818
5,resource_006,"Cherry / stone fruits, Peach",35,809.400000
6,resource_007,Corn / maize,17,765.588235
7,resource_008,Grape,24,836.041667
8,resource_009,Orange / citrus,30,647.766667
9,resource_010,Bell pepper,17,716.411765


In [6]:
embedding = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

if PERSIST_DIR.exists():
    shutil.rmtree(PERSIST_DIR)

chunk_ids = [chunk.metadata["chunk_id"] for chunk in chunks]

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    ids=chunk_ids,
    collection_name=COLLECTION_NAME,
    persist_directory=str(PERSIST_DIR),
)

print(f"Persisted Chroma collection `{COLLECTION_NAME}` to {PERSIST_DIR}")
print(f"Indexed chunks: {vector_db._collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Persisted Chroma collection `plant_disease_resources` to /work/AAI_project/rag/chroma_db
Indexed chunks: 1396


In [7]:
def show_results(query: str, k: int = 4) -> None:
    print(f"Query: {query}\n")
    results = vector_db.similarity_search_with_score(query, k=k)
    for rank, (document, score) in enumerate(results, start=1):
        metadata = document.metadata
        preview = document.page_content.replace("\n", " ")[:500]
        print(f"--- Result {rank} | score={score:.4f} ---")
        print("Resource:", metadata.get("resource_id"))
        print("Labels:", metadata.get("labels"))
        print("Chunk:", metadata.get("chunk_index"))
        print("Source:", metadata.get("source_url"))
        print(preview)
        print()


show_results("How do I manage tomato early blight?")
show_results("What are symptoms of cedar apple rust?")

Query: How do I manage tomato early blight?

--- Result 1 | score=0.3914 ---
Resource: resource_041
Labels: Potato early blight, Tomato early blight
Chunk: 6
Source: https://extension.umn.edu/disease-management/early-blight-tomato-and-potato
## Managing early blight on farms  Early blight typically appears in Minnesota in mid to late June. The exact timing varies from year to year, so scout regularly in order to begin managing the disease as soon as it appears.  - There are many resistant tomato cultivars available, often designated with an "EB" in seed catalogs.  - There is an extensive list of resistant cultivars on Cornell University's vegetable pathology website .  - Resistant varieties are not immune to early blight. However, i

--- Result 2 | score=0.4112 ---
Resource: resource_041
Labels: Potato early blight, Tomato early blight
Chunk: 7
Source: https://extension.umn.edu/disease-management/early-blight-tomato-and-potato
- Fertilize properly to maintain vigorous plant growth. Do 

The vector database is ready in `rag/chroma_db`. Later notebooks or agents can reload it with the same embedding model and collection name.